# Homework — Image Degradation & Restoration

**Course:** Medical Image Analysis & Processing (MIAP)


## Learning objectives

1. Recognise the most common types of degradation in medical images.
2. **Simulate** image degradation: Gaussian / Poisson / salt-and-pepper / speckle / periodic noise, motion blur, low contrast.
3. **Restore** degraded images with linear filters, the median filter, the Wiener deconvolution filter, and histogram-based contrast enhancement.
4. Evaluate restoration quality both visually and quantitatively (PSNR / SSIM).

## Dataset

All seven test images live in the [data/](data/) folder next to this notebook.

| Variable | Modality | File |
|---|---|---|
| `mr_a`     | MR (head)        | `MR.1.2.246.352.221.4656383511908937034762430477230289593.dcm` |
| `mr_b`     | MR (head)        | `MR.1.2.246.352.221.46120785868456727797248075870236962707.dcm` |
| `ct_a`     | Cranial CT       | `1.3.6.1.4.1.5962.99.1.2786334768.1849416866.1385765836848.164.0.dcm` |
| `ct_b`     | Cranial CT       | `1.3.6.1.4.1.5962.99.1.2786334768.1849416866.1385765836848.201.0.dcm` |
| `cr_chest` | Chest X-ray (CR) | `1.2.276.0.7230010.3.1.4.8323329.1000.1517875165.878027.dcm` |
| `us_liver` | Ultrasound liver | `USliver2.png` |
| `pattern`  | Geometric / gray-step test pattern | `Test pattern.png` |
| `tg18`     | TG18 medical-display test image | `TG18-CH-2k-01.dcm` |


---
## 0. Setup

Run the next cell once to install dependencies. If you already have them, the install is a no-op.


In [ ]:
# !pip install pydicom opencv-python numpy matplotlib scikit-image scipy
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
import pydicom
from scipy.ndimage import convolve
from scipy.fft import fft2, ifft2

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 5)

DATA = "data"        # folder next to the notebook
np.random.seed(0)    # so your results are reproducible

## 1. Loading helpers

Two small helpers normalise every image to `float32` in `[0, 1]` so that all later code can ignore bit depth.


In [ ]:
def load_dicom(path):
    """Read a DICOM file and return a float32 image normalised to [0, 1]."""
    ds = pydicom.dcmread(path)
    img = ds.pixel_array.astype(np.float32)
    return (img - img.min()) / (img.max() - img.min() + 1e-12)

def load_png_gray(path):
    """Read a PNG as float32 grayscale in [0, 1]."""
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(path)
    return img.astype(np.float32) / 255.0

# --- load every image ---------------------------------------------------------
mr_a     = load_dicom(os.path.join(DATA, "MR.1.2.246.352.221.4656383511908937034762430477230289593.dcm"))
mr_b     = load_dicom(os.path.join(DATA, "MR.1.2.246.352.221.46120785868456727797248075870236962707.dcm"))
ct_a     = load_dicom(os.path.join(DATA, "1.3.6.1.4.1.5962.99.1.2786334768.1849416866.1385765836848.164.0.dcm"))
ct_b     = load_dicom(os.path.join(DATA, "1.3.6.1.4.1.5962.99.1.2786334768.1849416866.1385765836848.201.0.dcm"))
cr_chest = load_dicom(os.path.join(DATA, "1.2.276.0.7230010.3.1.4.8323329.1000.1517875165.878027.dcm"))
us_liver = load_png_gray(os.path.join(DATA, "USliver2.png"))
pattern  = load_png_gray(os.path.join(DATA, "Test pattern.png"))
tg18     = load_dicom(os.path.join(DATA, "TG18-CH-2k-01.dcm"))

# Display all of them
imgs   = [mr_a, mr_b, ct_a, ct_b, cr_chest, us_liver, pattern, tg18]
titles = ["MR a", "MR b", "CT a", "CT b",
          "Chest X-ray", "US liver", "Test pattern", "TG18-CH"]
fig, ax = plt.subplots(2, 4, figsize=(16, 8))
for a, im, t in zip(ax.ravel(), imgs, titles):
    a.imshow(im, cmap="gray"); a.set_title(f"{t} {im.shape}"); a.axis("off")
fig.tight_layout(); plt.show()

---
## Task 1 — Implement five noise models

Implement each of the five noise functions below. They must:
* take a float32 image in `[0, 1]`,
* return a noisy float32 image still clipped to `[0, 1]`.

For each function, write **one sentence** explaining where this kind of noise actually appears in real medical images.


In [ ]:
def add_gaussian_noise(image, mean=0.0, std=0.05):
    """Additive zero-mean Gaussian noise of standard deviation `std`."""
    # TODO
    noise = np.random.normal(mean, std, image.shape)
    noisy = image + noise

    return np.clip(noisy, 0, 1).astype(np.float32)

def add_poisson_noise(image):
    """Photon-count (shot) noise — variance = signal.
    Hint: np.random.poisson(image * scale) / scale."""
    # TODO
    scale = 255.0
    noisy = np.random.poisson(image * scale) / scale

    return np.clip(noisy, 0, 1).astype(np.float32)

def add_salt_and_pepper_noise(image, prob=0.05):
    """Replace `prob/2` of the pixels with 0 (pepper) and `prob/2` with 1 (salt)."""
    # TODO
    noisy = image.copy()
    random_vals = np.random.rand(*image.shape)
    noisy[random_vals < prob/2] = 0
    noisy[random_vals > 1 - prob/2] = 1

    return np.clip(noisy, 0, 1).astype(np.float32)

def add_speckle_noise(image, variance=0.05):
    """Multiplicative noise:  out = image + image * N(0, variance)."""
    # TODO
    noise = np.random.normal(0, np.sqrt(variance), image.shape)
    noisy = image + image * noise

    return np.clip(noisy, 0, 1).astype(np.float32)

def add_periodic_noise(image, frequency=20, amplitude=0.1):
    """Add a horizontal sinusoidal pattern (simulates electrical interference)."""
    # TODO
    rows, cols = image.shape[:2]

    x = np.arange(cols)
    sinusoid = amplitude * np.sin(2 * np.pi * frequency * x / cols)

    pattern = np.tile(sinusoid, (rows, 1))
    if image.ndim == 3:
        pattern = pattern[:, :, np.newaxis]

    noisy = image + pattern
    return np.clip(noisy, 0, 1).astype(np.float32)

Gaussian Noise: This appears as thermal and electronic noise in the amplifier circuits and receiver coils of MRI scanners and CT detectors.

Poisson Noise: This arises from the discrete, statistical nature of photon emission and detection in low-dose CT scans, PET, and SPECT imaging.

Salt and Pepper Noise: This occurs due to dead pixels, analog-to-digital converter errors, or data transmission losses in digital X-ray detectors.

Speckle Noise: This creates the characteristic granular pattern in Ultrasound images caused by the constructive and destructive interference of backscattered acoustic waves.

Periodic Noise: This manifests as repeating band-like artifacts due to power supply interference, electrical readout line issues, or mechanical vibrations during image acquisition.

**Question 1.1.** Which noise model best matches: 
* **Photon-limited** acquisitions (CT at very low dose, nuclear medicine)? Poisson noise.
* **Ultrasound** images?  Speckle noise.
* **MR** images at high field strength? Guassian noise.
* A faulty sensor read-out line? Periodic noise (electrical interference) or Salt and Pepper noise (if data pixels are entirely dropped).
 

---
## Task 2 — Apply each noise type to a real medical image

Pair each noise type with the image whose physics best matches it (you decide). Plot **original vs. noisy** side by side. A starter loop is provided.


In [ ]:
# Pick one image per noise type; you may justify your choice in a markdown cell below.
samples = {
    "Gaussian Noise"      : (mr_a,     add_gaussian_noise),
    "Poisson Noise"       : (ct_a,     add_poisson_noise),
    "Salt & Pepper Noise" : (cr_chest, add_salt_and_pepper_noise),
    "Speckle Noise"       : (us_liver, add_speckle_noise),
    "Periodic Noise"      : (pattern,     add_periodic_noise),
}

# TODO: for each (name, (image, fn)) plot original | noisy
# (you may want a 5x2 grid)

fig, axes = plt.subplots(5, 2, figsize=(12, 25))

for idx, (noise_name, (image, noise_fn)) in enumerate(samples.items()):

    noisy_image = noise_fn(image)
    
    axes[idx, 0].imshow(image, cmap='gray')
    axes[idx, 0].set_title(f"Original")
    axes[idx, 0].axis('off')
    
    axes[idx, 1].imshow(noisy_image, cmap='gray')
    axes[idx, 1].set_title(f"{noise_name}")
    axes[idx, 1].axis('off')

plt.tight_layout()
plt.show()


MRI -- Gaussian: MR reconstruction noise is well approximated by Gaussian in magnitude images.

CT/X‑ray -- Poisson: X‑ray photon detection follows Poisson statistics.

CR chest -- Salt & Pepper: Simulates faulty detector pixels or digital corruption.

Ultrasound -- Speckle: Caused by coherent wave interference.

Test pattern -- Periodic: Makes sinusoidal artifacts easy to observe in both spatial and frequency domains.

**Question 2.1.** Which noise type is *least* visible to your eye? Which is most disruptive for diagnosis? Why?


Least visible:

Poisson noise is generally the least visible to the human eye. Because Poisson noise is signal-dependent, its variance scales with the image intensity. In brighter regions where the signal is strong, the relative noise (SNR) is actually better, and in darker regions, our eyes are naturally less sensitive to the high-frequency variations it introduces, perceiving it simply as a subtle film grain.

Most disruptive:

Salt & Pepper noise and Periodic noise are the most disruptive for diagnosis.

Salt & Pepper noise completely overwrites pixel values with extreme black or white dots, directly destroying fine details (like tiny vessels in X-rays) which are critical for diagnosis.
Periodic noise is equally disruptive because it introduces strong, artificial geometric patterns (bands/stripes) that create false edges, potentially masking larger pathological structures and confusing both human readers and automated segmentation algorithms.

---
## Task 3 — Test pattern and noise histograms

The variable `pattern` (loaded from `Test pattern.png`) contains regions of constant intensity at several gray levels plus several hard edges — exactly the kind of stimulus that lets you read off noise statistics directly from a histogram.

Apply each of your five noise functions to the test pattern and plot **noisy image + intensity histogram** for each.


In [ ]:
# Quick look at the test pattern and its histogram so you can see the
# discrete intensity peaks that noise will smear out.
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(pattern, cmap="gray", vmin=0, vmax=1)
ax[0].set_title("Test pattern"); ax[0].axis("off")
ax[1].hist(pattern.ravel(), bins=256, color="gray")
ax[1].set_title("Histogram of clean test pattern")
ax[1].set_xlabel("intensity"); ax[1].set_ylabel("count")
plt.tight_layout(); plt.show()

In [ ]:
# TODO: for each noise function, plot the noisy pattern AND its histogram (256 bins).
# Aim for a 5 x 2 layout: rows = noise type; columns = image | histogram.
# Compare each noisy histogram to the clean one above.
noise_functions = [
    ("Gaussian Noise", add_gaussian_noise),
    ("Poisson Noise", add_poisson_noise),
    ("Salt & Pepper Noise", add_salt_and_pepper_noise),
    ("Speckle Noise", add_speckle_noise),
    ("Periodic Noise", add_periodic_noise),
]

fig, ax = plt.subplots(5, 2, figsize=(12, 20))

for i, (name, func) in enumerate(noise_functions):
    noisy = func(pattern)

    ax[i, 0].imshow(noisy, cmap="gray", vmin=0, vmax=1)
    ax[i, 0].set_title(f"{name} (image)")
    ax[i, 0].axis("off")

    ax[i, 1].hist(noisy.ravel(), bins=256, color="gray")
    ax[i, 1].set_title(f"{name} (histogram)")
    ax[i, 1].set_xlabel("intensity")
    ax[i, 1].set_ylabel("count")

plt.tight_layout()
plt.show()


**Question 3.1.** From the histograms alone, can you tell Gaussian noise apart from speckle noise?

Based on these histograms, yes, we can tell them apart. In Gaussian noise, every peak has the same width. In Speckle noise, the peaks get progressively wider and flatter as we move toward the right (higher intensity).

**Question 3.2.** What pattern do you see in the histogram of *salt-and-pepper* noise?

 The Salt & Pepper histogram shows the original signal peaks preserved perfectly, but with added “impulses” (spikes) at the extreme boundaries (0 and 1).


---
## Task 4 — Motion blur and low-contrast degradation

Implement a motion-blur model and a contrast-reduction model.


In [ ]:
def add_motion_blur(image, kernel_size=15, angle=0.0):
    """Convolve `image` with an oriented box-line kernel.
    Return (blurred_image, kernel) so the kernel can be re-used by the deblur step.
    Hint: build a horizontal line kernel and rotate it with cv2.warpAffine + cv2.getRotationMatrix2D."""
    # TODO
    kernel = np.zeros((kernel_size, kernel_size))
    center = (kernel_size - 1) // 2
    kernel[center, :] = 1.0
    
    rot_mat = cv2.getRotationMatrix2D((kernel_size / 2 - 0.5, kernel_size / 2 - 0.5), angle, 1.0)
    kernel = cv2.warpAffine(kernel, rot_mat, (kernel_size, kernel_size))
    
    kernel /= np.sum(kernel)

    blurred_image = cv2.filter2D(image, -1, kernel)
    return blurred_image.astype(np.float32), kernel

def reduce_contrast(image, factor=0.3):
    """Squeeze pixel intensities towards the image mean by `factor` in (0,1].
    factor=1 -> unchanged, factor=0 -> uniform gray."""
    # TODO
    img_mean = np.mean(image)
    low_contrast = img_mean + factor * (image - img_mean)
    return np.clip(low_contrast, 0, 1).astype(np.float32)

# TODO: produce
#   blurred, psf = add_motion_blur(mr_b, kernel_size=15, angle=30)
#   low_contrast = reduce_contrast(ct_b, factor=0.3)
# and plot the four images in a 2x2 grid.

blurred, psf = add_motion_blur(mr_b, kernel_size=15, angle=30)
low_contrast = reduce_contrast(ct_b, factor=0.3)

fig, ax = plt.subplots(2, 2, figsize=(10, 10))

ax[0, 0].imshow(mr_b, cmap="gray", vmin=0, vmax=1)
ax[0, 0].set_title("Original MR (mr_b)")
ax[0, 0].axis("off")

ax[0, 1].imshow(blurred, cmap="gray", vmin=0, vmax=1)
ax[0, 1].set_title("Motion Blurred")
ax[0, 1].axis("off")

ax[1, 0].imshow(ct_b, cmap="gray", vmin=0, vmax=1)
ax[1, 0].set_title("Original CT (ct_b)")
ax[1, 0].axis("off")

ax[1, 1].imshow(low_contrast, cmap="gray", vmin=0, vmax=1)
ax[1, 1].set_title(f"Reduced Contrast")
ax[1, 1].axis("off")

plt.tight_layout()
plt.show()

**Question 4.1.** Why does motion blur reduce **only** the high-frequency content of the image? What does that tell you about how to reverse it (Task 6)?


---
## Task 5 — Denoising filters

Reduce the noise from Task 2 with three classical filters. Choose the right pair of (noise, filter) and **explain** your choice.

| Noise type           | Best linear/non-linear filter |
|----------------------|-------------------------------|
| Gaussian             |   Gaussian filter             |
| Poisson              |   Gaussian filter             |
| Salt & Pepper        |   Median filter               |
| Speckle              |   Mean filter                 |


In [ ]:
def mean_filter(image, ksize=5):
    return cv2.blur((image * 255).astype(np.uint8), (ksize, ksize)).astype(np.float32) / 255

def gaussian_filter(image, ksize=5, sigma=0):
    return cv2.GaussianBlur((image * 255).astype(np.uint8), (ksize, ksize), sigma).astype(np.float32) / 255

def median_filter(image, ksize=5):
    if ksize % 2 == 0:
        ksize += 1
    return cv2.medianBlur((image * 255).astype(np.uint8), ksize).astype(np.float32) / 255

# TODO:
# 1. Recreate three noisy images: gaussian_noisy, poisson_noisy, sp_noisy.
# 2. Apply the filter you think is best to each.
# 3. Plot a 3x3 grid: rows = noise type;  columns = noisy | mean | gaussian | median.
#    (Yes that's 3x4 — pick whichever layout you prefer.)

gaussian_noisy = add_gaussian_noise(mr_a, std=0.05)
poisson_noisy  = add_poisson_noise(ct_a)
sp_noisy       = add_salt_and_pepper_noise(cr_chest, prob=0.05)

images = {
    "Gaussian noise": {
        "Noisy": gaussian_noisy,
        "Mean": mean_filter(gaussian_noisy),
        "Gaussian": gaussian_filter(gaussian_noisy),
        "Median": median_filter(gaussian_noisy),
    },
    "Poisson noise": {
        "Noisy": poisson_noisy,
        "Mean": mean_filter(poisson_noisy),
        "Gaussian": gaussian_filter(poisson_noisy),
        "Median": median_filter(poisson_noisy),
    },
    "Salt & Pepper noise": {
        "Noisy": sp_noisy,
        "Mean": mean_filter(sp_noisy),
        "Gaussian": gaussian_filter(sp_noisy),
        "Median": median_filter(sp_noisy),
    }
}

fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for row, (noise_name, imgs) in enumerate(images.items()):
    for col, (title, img) in enumerate(imgs.items()):
        axes[row, col].imshow(img, cmap="gray", vmin=0, vmax=1)
        axes[row, col].set_title(f"{noise_name}\n{title}")
        axes[row, col].axis("off")

plt.tight_layout()
plt.show()

**Question 5.1.** Why does the **median** filter excel at salt-and-pepper noise but not at Gaussian noise?

The median filter excels at salt‑and‑pepper noise because this noise consists of isolated extreme outliers (pixels randomly set to 0 or 1), and the median operation naturally ignores such outliers when selecting the middle value in a local neighborhood. As a result, impulse pixels are removed while edges and underlying structures are largely preserved. 

In contrast, Gaussian noise affects every pixel with small continuous fluctuations rather than isolated extremes, so there are no outliers for the median to reject. Applying a median filter to Gaussian noise therefore does not efficiently reduce the noise and can instead distort fine image details and smooth gradual intensity variations.

**Question 5.2.** What is the price you pay for using a larger smoothing kernel?

The price of using a larger smoothing kernel is the loss of image detail and spatial resolution. While increasing the kernel size improves noise suppression by averaging over more pixels, it also blurs edges, removes fine textures, and can eliminate small but diagnostically important structures such as vessels or subtle lesions. In frequency terms, a larger kernel removes more high‑frequency content, which cannot be recovered later. Thus, there is an inherent trade‑off between noise reduction and preservation of anatomical detail.

---
## Task 6 — Wiener deconvolution for motion blur

Recall that motion blur is a *convolution* with a known PSF, so it can be undone in the Fourier domain. Implement a Wiener filter:

$$
\hat{F}(u, v) \;=\; \frac{H^{*}(u,v)}{|H(u,v)|^{2} + K} \; G(u, v)
$$

where $G$ is the FFT of the blurred image, $H$ is the FFT of the PSF zero-padded to the image size, $H^{*}$ is its complex conjugate, and $K$ is a small regularisation constant (the noise-to-signal power ratio).


In [ ]:
from scipy.fft import fft2, ifft2, ifftshift

def wiener_deblur(blurred, psf, noise_power=1e-2):
    """Wiener deconvolution."""
    # TODO
    img_h, img_w = blurred.shape
    psf_h, psf_w = psf.shape
    
    pad_h = (img_h - psf_h) // 2
    pad_w = (img_w - psf_w) // 2

    padded_psf = np.pad(psf, ((pad_h, img_h - psf_h - pad_h), (pad_w, img_w - psf_w - pad_w)), mode='constant')
    padded_psf = ifftshift(padded_psf)
    
    H = fft2(padded_psf)
    G = fft2(blurred)
    
    # Wiener filter
    H_conj = np.conj(H)
    H_mag_sq = np.abs(H)**2
    F_hat = (H_conj / (H_mag_sq + noise_power)) * G
    
    deblurred = np.real(ifft2(F_hat))
    
    return np.clip(deblurred, 0, 1)

# TODO: blur ct_a with kernel_size=15 angle=30, then deblur and plot
# original | blurred | deblurred. Try several values of `noise_power`.

blurred_img, psf = add_motion_blur(ct_a, kernel_size=15, angle=30.0)

noise_powers = [1e-1, 1e-3, 1e-5, 0]

fig, axes = plt.subplots(1, 2 + len(noise_powers), figsize=(20, 5))

axes[0].imshow(ct_a, cmap='gray', vmin=0, vmax=1)
axes[0].set_title("Original")
axes[0].axis('off')

axes[1].imshow(blurred_img, cmap='gray', vmin=0, vmax=1)
axes[1].set_title("Blurred")
axes[1].axis('off')

for i, K in enumerate(noise_powers):

    deblurred = wiener_deblur(blurred_img, psf, noise_power=K)
    axes[2+i].imshow(deblurred, cmap='gray', vmin=0, vmax=1)
    axes[2+i].set_title(f"Deblurred (K={K})")
    axes[2+i].axis('off')

plt.tight_layout()
plt.show()

**Question 6.1.** What happens when you set `noise_power = 0`? Why?

Setting the noise power 𝐾 to 0 reduces the Wiener filter to a standard inverse filter:
$$\^𝐹(𝑢,𝑣)=𝐺(𝑢,𝑣)/𝐻(𝑢,𝑣)$$

Because motion blur suppresses certain high frequencies, the blur transfer function H(u,v) approaches 0 at those frequencies. Dividing by values near zero amplifies any minor noise or numerical inaccuracies in G(u,v) enormously. This overwhelms the true signal during the inverse FFT, producing a deblurred image dominated by noise and severe ringing artifacts.


**Question 6.2.** What goes wrong if you use the **wrong** PSF (e.g. the right kernel size but the wrong angle)?

If the wrong PSF is used, the Wiener filter attempts to divide the image frequencies by the wrong blur model. The algorithm will amplify and suppress the wrong spatial frequencies. Consequently, the original motion blur is not removed, and severe structural artifacts (such as “ghosting”, heavy directional ringing, and doubling of edges) are introduced in the direction of the incorrect angle.


---
## Task 7 — Contrast enhancement

Apply three contrast-enhancement methods to the **TG18-CH** medical-display test image (`tg18`) and compare the result both visually and via histograms. The same code should also work on `cr_chest` — try it on both if you have time.


In [ ]:
def to_uint8(img01):
    return (np.clip(img01, 0, 1) * 255).astype(np.uint8)

tg18_uint8 = to_uint8(tg18)

def histogram_equalisation(image_uint8):
    # TODO: cv2.equalizeHist
    return cv2.equalizeHist(image_uint8)

def adaptive_histogram_equalisation(image_uint8, grid_size=8, clip_limit=40.0):
    # TODO: cv2.createCLAHE with a *very large* clip_limit -> classical AHE
    ahe_obj = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(grid_size, grid_size))
    return ahe_obj.apply(image_uint8)

def clahe(image_uint8, grid_size=8, clip_limit=3.0):
    # TODO: cv2.createCLAHE with a small clip_limit
    clahe_obj = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(grid_size, grid_size))
    return clahe_obj.apply(image_uint8)

# TODO: apply HE / AHE / CLAHE to tg18_uint8 and build a 4-row figure
# (original | HE | AHE | CLAHE), each row showing image + histogram
# (use 256 bins, range [0,256]).

img_he = histogram_equalisation(tg18_uint8)
img_ahe = adaptive_histogram_equalisation(tg18_uint8)
img_clahe = clahe(tg18_uint8)
    
images = [
    ("Original", tg18_uint8),
    ("Global HE", img_he),
    ("AHE (High clip_limit)", img_ahe),
    ("CLAHE", img_clahe)
]
    
fig, axes = plt.subplots(4, 2, figsize=(12, 16))

for i, (title, img) in enumerate(images):

    axes[i, 0].imshow(img, cmap='gray', vmin=0, vmax=255)
    axes[i, 0].set_title(title)
    axes[i, 0].axis('off')

    axes[i, 1].hist(img.ravel(), bins=256, range=[0, 256], color='black')
    axes[i, 1].set_title(f"{title} Histogram")
    axes[i, 1].set_xlim([0, 256])
    
plt.tight_layout()
plt.show()


**Question 7.1.** Look at the histogram of the original TG18 image. Why does global histogram equalisation often make medical images look *worse* despite spreading the histogram?

Global histogram equalisation often makes medical images look worse because it boosts contrast everywhere, including places with no useful detail. This amplifies noise, makes smooth areas look grainy, and can overly brighten or darken important structures. Even though the histogram becomes evenly spread, the image becomes less natural and harder to read.

**Question 7.2.** What does the `clip_limit` parameter of CLAHE physically control?

The clip_limit in CLAHE controls how much contrast each small region is allowed to gain. A low clip limit keeps changes gentle and reduces noise, while a high clip limit makes the image much stronger but can also amplify noise. It’s basically a “contrast safety cap.”

**Question 7.3.** TG18-CH is a *medical-display* test pattern. After CLAHE, are the gray-step ramps still monotonically increasing? Should they be?

After CLAHE, the TG18‑CH gray ramps may no longer increase smoothly from dark to bright. This happens because CLAHE adjusts contrast locally and can slightly reorder nearby gray levels. But for a display‑test pattern, the steps should stay monotonic, so CLAHE is not appropriate for evaluating display accuracy.

---
## Task 8 — Quantitative evaluation (bonus, recommended)

Measure restoration quality with two standard metrics:

* **PSNR** — peak signal-to-noise ratio,
* **SSIM** — structural similarity index.

Use `skimage.metrics.peak_signal_noise_ratio` and `skimage.metrics.structural_similarity`.


In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import pandas as pd
#
# TODO: for one image (e.g. ct_a):
# 1. degrade with Gaussian noise (std=0.05)
# 2. restore with mean / gaussian / median filters
# 3. report PSNR and SSIM of (original vs. noisy) and (original vs. restored)
# 4. summarise in a small table.


std_noise = 0.05
noisy_ct = add_gaussian_noise(ct_a, std=std_noise)

restored_mean = mean_filter(noisy_ct, ksize=5)
restored_gauss = gaussian_filter(noisy_ct, ksize=5)
restored_median = median_filter(noisy_ct, ksize=5)


def get_metrics(original, processed):

    p_val = psnr(original, processed, data_range=1.0)
    s_val = ssim(original, processed, data_range=1.0)
    return p_val, s_val


results = []

p, s = get_metrics(ct_a, noisy_ct)
results.append({"Method": "Noisy (Unfiltered)", "PSNR": p, "SSIM": s})

p, s = get_metrics(ct_a, restored_mean)
results.append({"Method": "Mean Filter", "PSNR": p, "SSIM": s})

p, s = get_metrics(ct_a, restored_gauss)
results.append({"Method": "Gaussian Filter", "PSNR": p, "SSIM": s})

p, s = get_metrics(ct_a, restored_median)
results.append({"Method": "Median Filter", "PSNR": p, "SSIM": s})

df_metrics = pd.DataFrame(results)

print("Quantitative Evaluation of Denoising on ct_a:")
display(df_metrics)



**Question 8.1.** Are PSNR and SSIM **always** correlated? Sketch a case where they disagree.

No, PSNR and SSIM are not always correlated.

PSNR is a purely mathematical metric based on absolute pixel-by-pixel differences (Mean Squared Error). SSIM, on the other hand, is designed to mimic human visual perception by looking at structural information, contrast, and luminance.

A case where they disagree:

Imagine taking an image and shifting it entirely to the right by just one or two pixels.

PSNR will drop drastically and report a very bad score because almost every single pixel no longer matches the exact pixel in the original image.
SSIM will remain relatively high because the structural elements (shapes, edges, and textures) of the anatomy are perfectly preserved, just slightly moved. To a human observer, the image quality hasn’t degraded, which SSIM captures better than PSNR.

Another classic example is slightly increasing the brightness of the entire image: PSNR drops heavily due to the global error, while SSIM correctly identifies that the internal structures are completely intact.

---
## What to hand in

* This notebook with every cell executed end-to-end.
* Short markdown answers under each question (1.1 - 8.1).
* A 200-300-word **conclusion** describing, for each of the four restoration
  scenarios you implemented, when it works well and when it fails.


$$CONCLUSION$$
**1. Linear Denoising (Mean & Gaussian Filters):** 
These spatial filters work well for suppressing continuous, randomly distributed noise (such as mild Gaussian or Poisson noise) in homogeneous regions of medical images. They fail, however, when preserving detail is critical; they indiscriminately blur high-frequency information (like fine anatomical edges) and fail completely at removing extreme outliers like Salt & Pepper noise, instead smearing the defective pixels across the image.

**2. Non-linear Denoising (Median Filter):** 
The median filter works exceptionally well for eliminating impulse artifacts (Salt & Pepper noise) caused by defective pixels or transmission errors, effectively removing outliers while preserving sharp structural boundaries. It fails when applied to heavy Gaussian noise or finely textured images (such as Ultrasound speckle), where it can wipe out diagnostic micro textures.

**3. Wiener Deconvolution (Deblurring):** 
This frequency-domain method works well for restoring images degraded by a known blur (like motion blur) in the presence of noise. By factoring in the Signal-to-Noise Ratio (SNR), it recovers sharpness without over-amplifying noise. It fails dramatically if the Point Spread Function (PSF) is incorrect—resulting in severe ghosting and ringing artifacts or if the noise power is underestimated (e.g., set to $0$), which reduces it to a naive inverse filter that destroys the image with high-frequency noise.

**4. Contrast Enhancement (CLAHE):** 
Contrast Limited Adaptive Histogram Equalization (CLAHE) works well for highlighting hidden details in low-contrast acquisitions (like specific CR or DR X-rays) by locally redistributing intensities without blowing out the global dynamic range. Global Histogram Equalization (HE), however, fails in almost all medical contexts by amplifying background noise and washing out details. Even CLAHE fails if its clip_limit is set too high or if it is applied to calibration images where an absolute, global luminance scale must be preserved.